# Load project
Basert på blant annet https://github.com/Tanawin1701d/vitisUnifiedTutorial/blob/main/part8b_testOnHw.ipynb og tidligere eksperimentering

In [18]:
# import the library
from pynq import Overlay     # import the overlay
from pynq import allocate    # import for CMA (contingeous memory allocation)
from pynq import DefaultIP   # import the ip connector library for extension
from pynq import Interrupt
import axi_master_driver
import asyncio
import numpy as np
import os
import subprocess
import re
import time

In [19]:
# create the overlay object
overlay = Overlay("system.bit")
help(overlay)
#overlay?

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay system.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    axi_vip_pltf         : pynq.overlay.DefaultIP
    myproject_axi_master_1 : axi_master_driver.HLS4ML_IP
    axi_intc_0           : pynq.overlay.DefaultIP
    ps_e                 : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    PSDDR                : Memory



In [20]:
# create an instance of the interrupt
my_interrupt = Interrupt('myproject_axi_master_1/interrupt')

In [21]:
# Load input from .npy file
x_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

input_array  = x_test.astype(np.float16) 
output_array = np.zeros(y_test.shape, dtype=np.float16)

# Allocate physically contiguous memory for input and output
input_buffer = allocate(shape=input_array.shape, dtype=np.float16)
output_buffer = allocate(shape=output_array.shape, dtype=np.float16)

# check input shape
print(f"input array shape {input_array.shape}")
print(f"output array shape {output_array.shape}")

input array shape (166000, 16)
output array shape (166000, 5)


In [22]:
# copy data to input buffer
np.copyto(input_buffer, input_array)
input_buffer.flush()

In [23]:
# get the ip and initialize the system
ip = overlay.myproject_axi_master_1  # Replace with your IP instance name
ip.set_input (0, input_buffer)
ip.set_output(0, output_buffer)
#ip.set_amt_query(input_array.shape[0]) # object has no attribute 'REG_ADDR_AMT_QUERY'
ip.prepare_intr()

input gmem_in0_ptr_fc1_input will be set to addr: 0x3ba00000 with elements: 2656000
output gmem_out0_ptr_layer13_out will be set to addr: 0x38600000 with elements: 830000
prepare your interrupt
global interrupt enable register
enable gie successful
ap_done interrupt enable register
enable ap_done interrupt successful
ap_done register clear
clear ap_done interrupt successful
----------------------


In [24]:
async def wait_for_acc():
    print("starting the accelerator")
    ip.ctrl_start()
    print("waiting for the accelerator to finish")
    await my_interrupt.wait()
    print("accelerator has finished")


# Inference

In [25]:
#### get event loop from asyncio
loop = asyncio.get_event_loop()

In [31]:
task = loop.create_task(wait_for_acc())
loop.run_until_complete(task)

starting the accelerator
waiting for the accelerator to finish
accelerator has finished


In [32]:
output_buffer.invalidate()

In [34]:
print(input_buffer)
print(output_buffer)

[[-0.1195   0.4062  -1.041   ...  0.4087  -1.02    -0.1802 ]
 [ 0.3088   0.2272  -1.156   ...  1.867   -1.232   -1.195  ]
 [-1.266    0.6675   1.409   ...  0.5825   1.31     1.757  ]
 ...
 [ 1.1455  -0.467   -0.3691  ... -1.184    0.05066 -0.6875 ]
 [-0.10474  0.3005   1.344   ... -0.521    1.38     0.235  ]
 [-0.9126   0.7295   0.1798  ... -0.2944   0.09424  0.558  ]]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [29]:
# convert it to numpy array
print("we got output shape:", output_buffer.shape)
outNp = np.array(output_buffer)

we got output shape: (166000, 5)


In [30]:
# save it to .npy file
np.save("out_hw.npy", outNp)